# PR Agent Eval

In this notebook, you examine two PR review datapoints from [awslabs/mcp](https://github.com/awslabs/mcp). You learn how each part of a datapoint is made, and why. Then you select the metrics, and you run Codex as a PR review agent in Harbor.

The datapoints are in `datasets/awslabs-mcp-pr-review/`. We made them before the workshop. It is not necessary to make datapoints in this notebook.

### Procedure

1. Learn how the datapoints were made.
2. Examine the two datapoints.
3. Examine the task files and do a leakage scan.
4. Select the metrics.
5. Run the oracle and Codex in Harbor.
6. Calculate the results.
7. Look at the results in Harbor view.

An editor that you open from the Dock or Finder does not read your shell profile, so the kernel cannot find `docker`, `codex`, or `harbor`. This cell adds your shell `PATH` to the kernel.

The cell also loads your settings from `.env`. To use a named AWS profile or a different Region, copy `.env.example` to `.env` and set `AWS_PROFILE` and `AWS_REGION`. Then restart the kernel or run this cell again.

In [ ]:
from env_setup import fix_path, load_env; fix_path(); load_env()

In [ ]:
import json
import os
import re
import shutil
import subprocess
import tomllib
from collections import Counter
from pathlib import Path

DATASET_DIR = Path("datasets/awslabs-mcp-pr-review")
TASKS_DIR = DATASET_DIR / "tasks"
JOBS_DIR = Path("jobs")
REJECT_TASK = TASKS_DIR / "awslabs__mcp__pr4615-reject"
APPROVE_TASK = TASKS_DIR / "awslabs__mcp__pr4121-approve"
TASK_DIRS = [REJECT_TASK, APPROVE_TASK]

# Same Bedrock settings as notebook 01. The task.toml files allow only the Bedrock endpoint in
# us-east-1. If you change AWS_REGION, also change allowed_hosts in each task.toml.
CODEX_MODEL = "openai.gpt-5.5"
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")  # from .env, if you set it

def read_toml(path: Path) -> dict:
    return tomllib.loads(path.read_text())

def read_json(path: Path):
    return json.loads(path.read_text())

## 1. How the Datapoints Were Made

We used [sample-code-eval-harvester](https://github.com/aws-samples/sample-code-eval-harvester) to make the datapoints. It is a CLI named `eval-harvest`. A coding agent runs the CLI: the CLI collects the facts from git and GitHub, and the agent makes the decisions. The procedure has these steps:

| Step | Command | Who does it | Result |
|------|---------|-------------|--------|
| 1 | `init` | CLI | A dataset directory. A rubric and a risk map, from the conventions of the repository. |
| 2 | `survey` | CLI | A list of the PRs that can become datapoints, and the reason that the other PRs cannot. |
| 3 | `capture` | CLI | A candidate file for each PR: the review iterations, the patches, the review comments, and the reviews. |
| 4 | `brief`, `show`, `annotate` | Agent | A label for each review comment. A finding with a severity for each defect. A risk level for the change. |
| 5 | `emit` | CLI | A Harbor task directory with a sealed environment. |
| 6 | `verify` | CLI | Checks that the datapoint is correct, and that the agent cannot see the answer. |

In step 4, the agent gives one of these labels to each review comment. Only defects go into the oracle.

| Label | Meaning | In the oracle |
|-------|---------|---------------|
| **Defect** | The reviewer blocks the PR for this problem | Yes, with severity `low`, `medium`, or `high` |
| **Nit** | Style or preference | No |
| **Question** | A request for information | No |
| **Approval** | A sign-off with no finding | No |
| **Bot** | A comment from an automated tool | No |

> **Note:** Do not label a nit as a defect. If you do, the eval rewards an agent that reports nits.

To select the PRs and make the datapoints takes time and judgment. Thus this workshop gives you the result. You can use the same tool on your own repository.

## 2. Examine the Two Datapoints

The dataset has a reject datapoint and an approve datapoint. Both changes are security-relevant. One changes a guard against SQL that mutates a database. The other changes TLS certificate verification.

| | Reject datapoint | Approve datapoint |
|---|---|---|
| PR | [#4615](https://github.com/awslabs/mcp/pull/4615) | [#4121](https://github.com/awslabs/mcp/pull/4121) |
| Server | `mysql-mcp-server` | `eks-mcp-server` |
| Change | Read-only mode also rejects SQL statements that start with `DO`, `START`, `SHUTDOWN`, and other mutating verbs | The Kubernetes client accepts EKS cluster CAs that have no Authority Key Identifier |
| Version under review | Iteration 0. A reviewer requested changes on this version. | Iteration 1. This is the final version. Two reviewers approved it. |
| Expected verdict | `block` | `approve` |
| Oracle | 1 high, 2 medium, and 3 low findings | No findings |
| Risk (structural, classified) | high, high | medium, high |

### Why we selected these PRs

**PR 4615 (reject).** The PR tries to close a bypass of the read-only guard. A reviewer found that the bypass is still open: a MySQL comment such as `#x` before the verb hides the verb from the new check. This is a clear high-severity defect. The reviewer also gave a test that shows the defect. A good review agent must find it.

**PR 4121 (approve).** EKS makes cluster CAs without an Authority Key Identifier. OpenSSL 3 rejects these CAs when the `VERIFY_X509_STRICT` flag is set. The fix removes only that flag. Certificate verification stays required (`CERT_REQUIRED`), and the client still verifies against the EKS CA. The only review comment asked for a test of a second code branch. The final version adds that test, and two reviewers approved it. Thus the oracle has no findings.

The approve datapoint also tests judgment. The change edits TLS settings, thus it looks dangerous. An agent that blocks each change that looks dangerous fails this datapoint. A good reviewer sees that verification is still on.

An agent that blocks all PRs gets a perfect score on reject datapoints. The approve datapoint finds this problem.

The risk map does not include `src/eks-mcp-server/`, thus its structural risk is the default, `medium`. The classified risk is `high`, because the change edits certificate verification. `task.toml` records this disagreement as `change_risk_disagreement = true`.

### Make sure that an approval is correct

A human approval does not prove that the code is correct. A reviewer can approve a change that has a defect. Before you use an approved PR as an approve datapoint, examine the later commits to the same files. If a later commit fixes the same code, the approval was probably wrong.

An earlier version of this dataset used [#4268](https://github.com/awslabs/mcp/pull/4268) as the approve datapoint. That PR made an IAM policy guard stronger, and two reviewers approved it. Codex blocked it: `iam:*` on `arn:aws:iam::*:role/*` was not blocked by the guard. Ten weeks later, [#4701](https://github.com/awslabs/mcp/pull/4701) closed this gap. Thus the agent was correct, and the datapoint was wrong. For PR 4121, no later commit changes the code.

A reject datapoint and an approve datapoint can come from the same PR: the version that the reviewer rejected, and the version that the reviewer approved. They can also come from different PRs, as in this dataset.

The candidate files in `datasets/awslabs-mcp-pr-review/candidates/` record the data from step 3 and the decisions from step 4. This cell shows a summary.

In [ ]:
for pr in (4615, 4121):
    cand = read_json(DATASET_DIR / "candidates" / f"pr-{pr}.json")
    states = Counter(v["state"] for v in cand["review_verdicts"])
    labels = Counter(c["classification"] for c in cand["comments"])
    print(f"PR {pr}: {len(cand['iterations'])} iterations")
    print(f"  reviews:        {dict(states)}")
    print(f"  comment labels: {dict(labels)}")
    print(f"  findings:       {Counter(f['severity'] for f in cand['findings']) or 'none'}")
    risk = cand["change_risk"]
    print(f"  risk:           structural {risk['risk_structural']}, classified {risk['risk_classified']}")
    print(f"                  {risk['risk_classified_rationale']}")
    print()

Each finding records the comments that it comes from, the evidence for its severity, and the rubric rule that sets the severity. This is the high-severity finding of PR 4615:

In [ ]:
cand = read_json(DATASET_DIR / "candidates" / "pr-4615.json")
high = next(f for f in cand["findings"] if f["severity"] == "high")
for key in ("statement", "severity", "severity_evidence", "severity_rationale", "comment_ids"):
    print(f"{key}: {high[key]}\n")

## 3. Examine the Tasks

Each datapoint is one Harbor task directory. The agent sees `instruction.md` and the container that `environment/` builds. Only the verifier sees `tests/` and `solution/`.

In [ ]:
def show_tree(root: Path):
    print(root.name + "/")
    for path in sorted(root.rglob("*")):
        depth = len(path.relative_to(root).parts)
        print("    " * depth + path.name + ("/" if path.is_dir() else ""))

for task_dir in TASK_DIRS:
    show_tree(task_dir)
    print()

### `task.toml`

`task.toml` sets the network for the agent and for the verifier. Make sure that:

- The agent `network_mode` is `allowlist`, and `allowed_hosts` contains only the Bedrock endpoint that Codex uses. The agent cannot reach GitHub.
- The verifier `environment_mode` is `separate`. The verifier runs in a different container, and the agent cannot read the oracle.
- The verifier has network access and AWS credentials. The judge in `score.py` calls Amazon Bedrock. Harbor replaces each `${AWS_...}` value with the value from your shell.

`[metadata.harvest]` records the expected verdict, the severities, and the risk. It is not visible to the agent.

In [ ]:
for task_dir in TASK_DIRS:
    cfg = read_toml(task_dir / "task.toml")
    env, verifier = cfg["environment"], cfg["verifier"]
    print(f"{task_dir.name}")
    print(f"  agent network:    {env['network_mode']} {env.get('allowed_hosts', [])}")
    print(f"  verifier:         {verifier['environment_mode']}, network {verifier['environment']['network_mode']}")
    print(f"  verifier env:     {sorted(verifier.get('env', {}))}")
    print(f"  metadata.harvest: {cfg['metadata']['harvest']}")
    print()

    assert env["network_mode"] == "allowlist", "The agent network must be an allowlist."
    assert all("bedrock" in host for host in env["allowed_hosts"]), "Allow only the Bedrock endpoint."
    assert not any("github" in host for host in env["allowed_hosts"]), "The agent must not reach GitHub."
    assert verifier["environment_mode"] == "separate", "The verifier must be in a separate environment."
print("task.toml checks passed.")

### `instruction.md`

The instruction must be the same for all tasks. It must not refer to the PR, the reviewer, or the findings. If it does, the agent can learn the answer from the instruction.

The instruction tells the agent to write its findings to `/logs/artifacts/findings.json`. Harbor copies this file to the verifier.

In [ ]:
instructions = {t.name: (t / "instruction.md").read_text() for t in TASK_DIRS}
assert len(set(instructions.values())) == 1, "The instructions are not the same for all tasks."
print(instructions[REJECT_TASK.name])

### `environment/Dockerfile`

The Dockerfile builds the container that the agent works in. Make sure that it does these steps:

1. Clones only the base commit, the commit before the change.
2. Removes the remote and the remote refs.
3. Expires the reflog and runs garbage collection. After this step, only one commit is in the repository.
4. Applies `change.patch` as uncommitted changes. The agent reviews these changes.
5. Installs Python and the locked dependencies of the changed server. The agent can then run the tests and try inputs, as a human reviewer can. Without this step, the agent can only read the code.
6. Sets `HOME` to a directory that is not `/root`, and puts the Python environment on `PATH` for login shells too. Codex runs each command in a login shell.
7. Runs the full test suite of the changed server. If a test fails, the build fails. A test that fails before the agent starts makes the datapoint unfair: the agent cannot know if the change caused the failure.
8. Installs Codex.

The build has network access, but the agent has only its allowlist. Thus all the software must be installed at build time. The agent cannot install packages when it runs.

In [ ]:
print((REJECT_TASK / "environment" / "Dockerfile").read_text())

### `tests/oracle.json`

The oracle is the correct answer. For each finding, make sure that:

- It is a defect, not a nit.
- It has a severity: `low`, `medium`, or `high`. Only `high` findings block a merge.
- Its location touches a line that `change.patch` adds. The agent can find only what the change contains.

A location is a span of lines. An agent finding passes the location step if its line is in a span of the oracle finding. Then the judge decides if the two findings describe the same problem. A span that is too narrow is unfair: a correct finding on the next line gets no credit. Thus the high finding of the reject task has two spans: the function that strips the comments and runs the scan, and the definition of the statement-start pattern. A reviewer can report the defect at any of these lines.

A `reference_only` finding is a real defect, but the agent does not have to find it. It counts in the overall coverage, but not in the required coverage.

Also make sure that `expected_verdict` is `block` for the reject task and `approve` for the approve task.

In [ ]:
def added_lines(patch: str) -> dict[str, set[int]]:
    """Return the line numbers that a unified diff adds, for each file."""
    added, path, line_no = {}, None, None
    for line in patch.splitlines():
        if line.startswith("+++ "):
            path = line[6:] if line.startswith("+++ b/") else None
            added.setdefault(path, set())
        elif m := re.match(r"@@ -\d+(?:,\d+)? \+(\d+)(?:,\d+)? @@", line):
            line_no = int(m.group(1))
        elif path and line_no is not None and not line.startswith(("---", "\\")):
            if line.startswith("+"):
                added[path].add(line_no)
                line_no += 1
            elif line.startswith(" "):
                line_no += 1
    return added

for task_dir in TASK_DIRS:
    oracle = read_json(task_dir / "tests" / "oracle.json")
    expected = read_toml(task_dir / "task.toml")["metadata"]["harvest"]["expected_verdict"]
    changed = added_lines((task_dir / "environment" / "change.patch").read_text())
    print(f"{task_dir.name}: expected_verdict={oracle['expected_verdict']}")

    assert oracle["expected_verdict"] == expected
    for f in oracle["findings"]:
        assert f["severity"] in ("low", "medium", "high")
        for loc in f["locations"]:
            span = set(range(loc["line_start"], loc["line_end"] + 1))
            assert span & changed.get(loc["path"], set()), f"Finding {f['id']} does not touch an added line."
        flag = " (reference only)" if f["reference_only"] else ""
        spans = ", ".join(f"{l['line_start']}-{l['line_end']}" if l["line_end"] != l["line_start"] else str(l["line_start"]) for l in f["locations"])
        print(f"  {f['severity']:<6} {f['locations'][0]['path'].split('/')[-1]}:{spans}{flag}")
        print(f"         {f['statement'][:110]}...")

    blocking = [f for f in oracle["findings"] if f["severity"] == "high"]
    assert (expected == "block") == bool(blocking), "Only a task with a high finding can expect `block`."
    print()
print("oracle.json checks passed.")

### The rubric and the risk map

`rubric.md` tells the agent that labels the comments (step 4 in section 1) how to set a severity. It starts from a standard review rubric. Then it adds the conventions of awslabs/mcp, from files such as `DESIGN_GUIDELINES.md` and `CONTRIBUTING.md`. These conventions are the rules of the team, which public benchmarks do not have.

Severity and risk are different. Severity tells how bad a finding is. Risk tells how dangerous the change is. Record the two values separately:

- **Structural risk.** `risk-map.toml` maps path prefixes to risk levels. For example, `src/iam-mcp-server/` is `high`. If a path is not in the map, the risk is `medium`.
- **Classified risk.** A model or a person sets the risk from the content of the change.

Use risk to decide if the agent can approve low-risk PRs without a human.

In [ ]:
print((DATASET_DIR / "rubric.md").read_text())
print((DATASET_DIR / "risk-map.toml").read_text())

### Leakage scan

Scan all files that the agent sees. Search for the PR numbers, the reviewer names, and text from the review comments. If the agent can see this data, the score measures search, not review.

First, put a canary token in the environment. Make sure that the scan finds it. If the scan does not find it, the scan has a defect.

In [ ]:
def agent_visible_files(task_dir: Path):
    """Files that the agent sees."""
    yield task_dir / "instruction.md"
    yield from (p for p in (task_dir / "environment").rglob("*") if p.is_file())

def leakage_scan(task_dir: Path, needles: list[str]) -> list[tuple[str, str]]:
    hits = []
    for path in agent_visible_files(task_dir):
        text = path.read_text(errors="ignore").lower()
        for needle in needles:
            if needle and needle.lower() in text:
                hits.append((str(path.relative_to(task_dir)), needle))
    return hits

def needles_for(pr: int) -> list[str]:
    """The PR number, the reviewer names, and the first 60 characters of each review comment."""
    cand = read_json(DATASET_DIR / "candidates" / f"pr-{pr}.json")
    needles = [f"#{pr}", f"pull/{pr}"]
    needles += sorted({v["author"] for v in cand["review_verdicts"]})
    needles += [c["body"][:60] for c in cand["comments"] if len(c["body"]) >= 60]
    return needles

for task_dir, pr in [(REJECT_TASK, 4615), (APPROVE_TASK, 4121)]:
    # Canary test: the scan must find a known token.
    canary = "CANARY-7f3a91"
    canary_file = task_dir / "environment" / "canary.txt"
    canary_file.write_text(canary)
    try:
        assert leakage_scan(task_dir, [canary]), "The scan did not find the canary. The scan has a defect."
    finally:
        canary_file.unlink()

    needles = needles_for(pr)
    hits = leakage_scan(task_dir, needles)
    print(f"{task_dir.name}: {len(needles)} needles, " + ("no leaks." if not hits else f"leaks: {hits}"))

## 4. Select the Metrics

For each task, the agent writes a list of findings. `tests/score.py` compares the findings to the oracle in two steps. First, it compares the file and the line. Then an LLM judge decides if the two findings describe the same problem. The judge is pinned in `tests/judge.toml`: one model, and temperature 0.

`score.py` writes these signals to the reward file:

| Signal | Question |
|--------|----------|
| `coverage_required` | What fraction of the high (blocking) findings did the agent find? |
| `coverage_all` | What fraction of all oracle findings did the agent find? |
| `precision_adjudicated` | What fraction of the agent findings did the judge match to an oracle finding? |
| `precision_strict` | What fraction of the agent findings have the same location as an oracle finding? No judge. |
| `uncredited` | How many agent findings match no oracle finding? |
| `credited_<severity>`, `oracle_<severity>` | How many findings of each severity did the agent find, and how many are in the oracle? |

Compare `precision_strict` and `precision_adjudicated`. If they are very different, the score depends much on the judge. Examine the judge.

### The verdict

The instruction tells the agent to request changes only when it finds a high-severity defect. Thus the predicted verdict comes from the findings file: `block` if a finding has the severity `high`, and `approve` if not.

### Combine the signals into pass or fail

Each task gets one result: **pass** or **fail**. Use one of these methods:

- **A heuristic.** This notebook uses this method: pass if the verdict is correct, and for a reject task, `coverage_required` is 1.0.
- **An LLM judge with a rubric.** The judge decides the gray areas.

Then calculate the dataset metrics:

| Metric | Question |
|--------|----------|
| **Accuracy** | What fraction of tasks did the agent pass? |
| **Block recall** | Of the PRs that must be blocked, what fraction did the agent block? |
| **Block precision** | Of the PRs that the agent blocked, what fraction must be blocked? |
| **False approve rate** | How frequently did the agent approve a bad PR? This error has the highest cost. |
| **False block rate** | How frequently did the agent block a good PR? |

Obey these rules:

1. If the agent finds a problem that the reviewer did not find, record it as **uncredited**. Do not record it as wrong. Reviewers do not find all defects.
2. Keep each signal in the reward file. The signals show why a task failed.

## 5. Run the Tasks in Harbor

### Make sure that Docker runs

In [ ]:
ok = shutil.which("docker") and subprocess.run(["docker", "info"], capture_output=True).returncode == 0
assert ok, "Docker is not installed or does not run. Do the setup checks in notebook 01."
print("Docker runs.")

### Build the task images

Build each image before you run an agent. This finds incorrect clones and patches that do not apply. It does not use LLM tokens. The first build downloads the base image, installs the Python dependencies, runs the test suite, and installs Codex. This can take some minutes.

The cell also builds the image of the egress proxy (see section 5, "Enforce the network allowlist"). Harbor runs trials at the same time. If the proxy image is not built before the first run, two trials try to build it at the same time, and one trial fails with `image "docker.io/library/harvest-env-egress-proxy:1": already exists`.

In [ ]:
# The egress proxy image, built one time before Harbor needs it.
r = subprocess.run(["docker", "build", "-q", "-t", "harvest-env-egress-proxy:1", "harvest_env/egress_proxy"],
                   capture_output=True, text=True, stdin=subprocess.DEVNULL)
print(f"{'OK ' if r.returncode == 0 else 'ERR'} egress proxy  {r.stdout.strip() or r.stderr.strip()[-300:]}")

for task_dir in TASK_DIRS:
    r = subprocess.run(["docker", "build", "-q", "-t", f"eval-{task_dir.name}", str(task_dir / "environment")],
                       capture_output=True, text=True, stdin=subprocess.DEVNULL)
    print(f"{'OK ' if r.returncode == 0 else 'ERR'} {task_dir.name}  {r.stdout.strip() or r.stderr.strip()[-300:]}")

### Check the git seal

In each image, `git log --all` must show one commit only, and `git remote` must show no remotes. `git status` must show the changes from `change.patch`.

In [ ]:
SEAL_CHECK = 'echo "$(git log --all --oneline | wc -l) $(git remote | wc -l) $(git status --porcelain | wc -l)"'

for task_dir in TASK_DIRS:
    out = subprocess.run(["docker", "run", "--rm", f"eval-{task_dir.name}", "sh", "-c", SEAL_CHECK],
                         capture_output=True, text=True, check=True).stdout
    commits, remotes, changed = map(int, out.split())
    print(f"{task_dir.name}: commits={commits} remotes={remotes} changed_files={changed}")
    assert commits == 1, "More than one commit is reachable. The agent can find later commits."
    assert remotes == 0, "The repository has a remote."
    assert changed > 0, "change.patch is not applied."
print("Seal checks passed.")

### Enforce the network allowlist

Harbor's Docker environment enforces `allowlist` with nftables. This needs a kernel option (`CONFIG_NFT_FIB_INET`) that Docker Desktop does not have. On Docker Desktop, Harbor does not run a task that needs an allowlist.

This module includes `harvest_env/`, an environment from sample-code-eval-harvester that enforces the allowlist in a different way:

- The agent container is attached only to an internal Docker network. This network has no route off the host.
- A proxy container (Squid) is on the internal network and on a usual network. The proxy is the only path out. It forwards traffic only to the hosts in `allowed_hosts`.
- The agent container gets `HTTPS_PROXY`, so Codex and the AWS SDK send their traffic through the proxy.

A process that does not use the proxy cannot reach the network. On a host where Harbor's nftables method works, the environment uses Harbor's method.

You give the environment to Harbor with `-e harvest_env.docker_allowlist:ProxyAllowlistDockerEnvironment`. Harbor imports it, thus the module directory must be on `PYTHONPATH`.

### Give AWS credentials to Harbor

Codex in the agent container and the judge in the verifier both call Amazon Bedrock. Harbor gives them the `AWS_*` values from its own environment. This cell gets your credentials with boto3 and puts them in the environment of the kernel. Harbor runs as a child process of the kernel, thus it gets them too.

If you use a named profile, set `AWS_PROFILE` in `.env` (see `.env.example`). At Amazon, this is usually your Isengard profile. A profile that uses `credential_process` needs its command on the kernel `PATH`. `fix_path()` at the start of the notebook adds your shell `PATH`.

In [ ]:
import boto3

# AWS_PROFILE comes from .env (see .env.example). If it is not set, boto3 uses the default credential chain.
profile = os.environ.get("AWS_PROFILE") or None
session = boto3.Session(profile_name=profile, region_name=AWS_REGION)
creds = session.get_credentials()
if creds is None:
    raise RuntimeError(
        f"boto3 found no AWS credentials. Profile used: {profile or 'none (default credential chain)'}.\n"
        f"Profiles that boto3 can see: {session.available_profiles}\n"
        "Set AWS_PROFILE in .env to one of these profiles. Then run the load_env cell and this cell again."
    )
frozen = creds.get_frozen_credentials()

os.environ["AWS_ACCESS_KEY_ID"] = frozen.access_key
os.environ["AWS_SECRET_ACCESS_KEY"] = frozen.secret_key
if frozen.token:
    os.environ["AWS_SESSION_TOKEN"] = frozen.token
os.environ["AWS_REGION"] = AWS_REGION

arn = session.client("sts").get_caller_identity()["Arn"]
print(f"Profile: {profile or 'default credential chain'}")
print("AWS credentials found for:", arn.split(":")[-1])

### The Harbor helper

This helper runs Harbor with the proxy environment. It writes the Harbor output to a log file in `jobs/` and shows the end of the log.

If a run fails with an authentication error, your temporary credentials expired. Run the credentials cell again, then run the Harbor cell again.

In [ ]:
HARBOR_ENV = "harvest_env.docker_allowlist:ProxyAllowlistDockerEnvironment"

def harbor_run(job_name: str, *args: str) -> Path:
    """Run all tasks in TASKS_DIR. Return the job directory."""
    job_dir = JOBS_DIR / job_name
    if job_dir.exists():
        shutil.rmtree(job_dir)  # a new run replaces the old results
    JOBS_DIR.mkdir(exist_ok=True)
    cmd = ["harbor", "run", "-p", str(TASKS_DIR), "-e", HARBOR_ENV,
           "-o", str(JOBS_DIR), "--job-name", job_name, "-y", *args]
    env = {**os.environ, "PYTHONPATH": os.pathsep.join(filter(None, [os.getcwd(), os.environ.get("PYTHONPATH")]))}
    log = JOBS_DIR / f"{job_name}.log"
    with log.open("w") as f:
        r = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, env=env)
    print("\n".join(log.read_text().splitlines()[-8:]))
    assert r.returncode == 0, f"harbor run failed. See {log}."
    return job_dir

def trial_rewards(job_dir: Path) -> dict[str, dict]:
    """Return the reward of each trial, by task directory name."""
    rewards = {}
    for result in job_dir.glob("*/result.json"):
        task = read_json(result)["task_name"].replace("/", "__")
        rewards[task] = read_json(result.parent / "verifier" / "reward.json")
    return rewards

### Test the scorer with the oracle

The Harbor `oracle` agent runs `solution/solve.sh`. This script submits the oracle as the agent output. The score must be perfect. If it is not perfect, repair the scorer before you continue.

This run uses no tokens for the agent. The judge in the verifier calls Bedrock one time for each oracle finding.

In [ ]:
oracle_job = harbor_run("oracle", "-a", "oracle")

rewards = trial_rewards(oracle_job)
for task, reward in sorted(rewards.items()):
    print(f"{task}: coverage_all={reward['coverage_all']} precision_strict={reward['precision_strict']} uncredited={reward['uncredited']}")

reject, approve = rewards[REJECT_TASK.name], rewards[APPROVE_TASK.name]
assert reject["coverage_all"] == 1.0 and reject["precision_adjudicated"] == 1.0, "The scorer did not credit the oracle."
assert approve["uncredited"] == 0, "The approve oracle submitted findings."
print("The scorer gives the oracle a perfect score.")

### Run Codex

Now Codex reviews each change. The flags do these things:

| Flag | Purpose |
|------|---------|
| `-a codex -m openai.gpt-5.5` | Run Harbor's Codex agent with the model on Bedrock |
| `--ak config=codex-bedrock.toml` | Give Codex the Bedrock provider settings. Harbor uploads this file to the container. |
| `--ak disable_web_search=true` | Turn off web search. The allowlist also blocks it. |
| `--ae AWS_...=${AWS_...}` | Give the AWS credentials to Codex. Harbor replaces each `${...}` with the value from its environment. |

Each trial uses approximately 300,000 to 600,000 input tokens. Most of them are cached.

### Number of trials

`N_ATTEMPTS` sets the number of trials for each task. The default is 1 (pass@1): one trial for each task, two trials in total.

The agent output is not the same each time, thus one trial is not sufficient to make a decision about an agent. More trials give a more stable result, for example pass@k or a mean across trials.

> **Warning:** Each trial starts two containers on your computer (the agent and the proxy), then a verifier container. Harbor runs up to 4 trials at the same time. With `N_ATTEMPTS = 5`, this notebook runs 10 trials and uses approximately 3 to 6 million input tokens. This uses much CPU, memory, and disk on a laptop. For more trials or more tasks, run Harbor on external compute, for example Amazon EC2 or Lambda MicroVMs.

In [ ]:
print(Path("codex-bedrock.toml").read_text())

In [ ]:
N_ATTEMPTS = 1  # trials for each task. Read the warning above before you increase this value.

codex_job = harbor_run(
    "codex",
    "-k", str(N_ATTEMPTS),
    "-a", "codex", "-m", CODEX_MODEL,
    "--ak", "config=codex-bedrock.toml",
    "--ak", "disable_web_search=true",
    "--ae", "AWS_ACCESS_KEY_ID=${AWS_ACCESS_KEY_ID}",
    "--ae", "AWS_SECRET_ACCESS_KEY=${AWS_SECRET_ACCESS_KEY}",
    "--ae", "AWS_SESSION_TOKEN=${AWS_SESSION_TOKEN:-}",  # empty for long-term access keys
    "--ae", f"AWS_REGION={AWS_REGION}",
)

## 6. Calculate the Results

For each trial, this cell reads the reward file, the findings that the agent wrote, and the expected verdict and risk from `task.toml`. Then it uses the heuristic from section 4.

In [ ]:
def load_rows(job_dir: Path) -> list[dict]:
    """Return one row for each trial."""
    rows = []
    for result_path in sorted(job_dir.glob("*/result.json")):
        trial_dir = result_path.parent
        result = read_json(result_path)
        task = result["task_name"].replace("/", "__")
        harvest = read_toml(TASKS_DIR / task / "task.toml")["metadata"]["harvest"]
        findings_path = trial_dir / "artifacts" / "logs" / "artifacts" / "findings.json"
        findings = read_json(findings_path) if findings_path.exists() else []
        rows.append({
            "task": task,
            "expected_verdict": harvest["expected_verdict"],
            "predicted_verdict": "block" if any(f.get("severity") == "high" for f in findings) else "approve",
            "risk": harvest["change_risk_classified"],
            "findings": findings,
            "input_tokens": (result.get("agent_result") or {}).get("n_input_tokens"),
            **read_json(trial_dir / "verifier" / "reward.json"),
        })
    return rows

def passed(r: dict) -> bool:
    if r["predicted_verdict"] != r["expected_verdict"]:
        return False
    return r["expected_verdict"] == "approve" or r["coverage_required"] == 1.0

def summarize(rows: list[dict]) -> dict:
    should_block = [r for r in rows if r["expected_verdict"] == "block"]
    did_block = [r for r in rows if r["predicted_verdict"] == "block"]
    clean = [r for r in rows if r["expected_verdict"] == "approve"]
    rate = lambda num, den: round(len(num) / len(den), 3) if den else None
    return {
        "tasks": len(rows),
        "accuracy": rate([r for r in rows if passed(r)], rows),
        "block_recall": rate([r for r in should_block if r["predicted_verdict"] == "block"], should_block),
        "block_precision": rate([r for r in did_block if r["expected_verdict"] == "block"], did_block),
        "false_approve_rate": rate([r for r in should_block if r["predicted_verdict"] == "approve"], should_block),
        "false_block_rate": rate([r for r in clean if r["predicted_verdict"] == "block"], clean),
    }

rows = load_rows(codex_job)
for r in rows:
    print(f"{'PASS' if passed(r) else 'FAIL'} {r['task']}")
    print(f"     verdict: expected {r['expected_verdict']}, predicted {r['predicted_verdict']}")
    print(f"     coverage_required={r['coverage_required']} coverage_all={r['coverage_all']} "
          f"precision_adjudicated={r['precision_adjudicated']} uncredited={r['uncredited']:.0f}")
    for f in r["findings"]:
        print(f"     - [{f.get('severity')}] {f.get('path', '').split('/')[-1]}:{f.get('line')} {f.get('statement', '')[:100]}")
print()
print(json.dumps(summarize(rows), indent=2))

### Examine each failure

The score tells you that the agent failed. The trajectory tells you why. Each trial has its trajectory in `jobs/codex/<trial>/agent/`.

**The results change from run to run.** The agent can find a defect in one run and miss it in the next. With two tasks and one trial for each, the accuracy can be only 0, 0.5, or 1. To get a more stable result, increase `N_ATTEMPTS`. A failure that occurs in each trial is a weakness. A failure that occurs only sometimes is variance.

**Use a coding agent to read the trajectories.** A trajectory has many commands. A coding agent (for example, Claude Code, Kiro, or Codex) can read it faster than you can. For example:

```text
Read the Harbor job in jobs/codex/. For each oracle finding that the agent missed, read
agent/codex.txt and tell me what the agent read, which inputs it tried, and why it decided
that there was no defect. Also tell me if an environment problem stopped it. Do not change files.
```

**Find the cause.** Each failure has one of these causes:

| Cause | Example | What to do |
|-------|---------|------------|
| **The agent misses a defect that it can find** | The agent read the guard, but it did not try the input that breaks it. | Use the eval to improve the agent: change its prompt, its tools, or its review procedure. Then run the eval again. |
| **The agent does not have the context of the reviewer** | The reviewer knew how a different service uses the code. The repository does not show this. | Give the PR agent more context before you use it, for example design documents, team rules, or the services that use the code. |
| **The datapoint is wrong** | The agent blocked an approve task, and a later commit fixed the same defect. | Correct the oracle, or select a different PR (see section 2). |
| **The environment is broken** | A command failed because a tool is missing. | Repair the Dockerfile, then build the images again. |
| **The scorer did not give credit** | A correct finding is one line outside the span of the oracle finding. | Make the span wider in `oracle.json`. |


### Results by risk level

Before the agent approves low-risk PRs without a human, examine the low-risk results only. Make sure that there are sufficient low-risk datapoints. This dataset has no low-risk datapoints. Thus it cannot tell you if the agent can approve low-risk PRs.

In [ ]:
for risk in sorted({r["risk"] for r in rows}):
    subset = [r for r in rows if r["risk"] == risk]
    print(risk, json.dumps(summarize(subset)))

## 7. Look at the Results in Harbor View

`harbor view` starts a web server that shows each job, each trial, the reward, and the full trajectory of the agent: each command that it ran and each message that it wrote. Use it to find why a task failed.

This cell starts the server in the background, and shows the URL. Open the URL in your browser. The first start can take some time, because Harbor builds the viewer.

In [ ]:
import time

viewer_log = JOBS_DIR / "harbor-view.log"
viewer = subprocess.Popen(["harbor", "view", str(JOBS_DIR), "--port", "8080-8089"],
                          stdout=viewer_log.open("w"), stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL)

# Wait until the server shows its URL.
urls = []
for _ in range(120):
    urls = re.findall(r"https?://[\w.:-]+", viewer_log.read_text())
    if urls or viewer.poll() is not None:
        break
    time.sleep(1)

if urls:
    print(f"Harbor view runs at {urls[0]}")
else:
    print(viewer_log.read_text()[-2000:])
    raise RuntimeError(f"harbor view did not start. See {viewer_log}.")

When you are done, run this cell to stop the server. You can also run `harbor view jobs` in a terminal in this directory.

In [ ]:
viewer.terminate()
viewer.wait(timeout=10)
print("Harbor view stopped.")

## Summary

- The datapoints come from the PR history: the code before the change, the change, and the review.
- A reject datapoint and an approve datapoint together find an agent that blocks all PRs.
- Only defects go into the oracle. Do not include nits.
- Seal the environment: one commit, no remote, and a network allowlist without GitHub.
- Run the oracle first. The score must be perfect.
- Combine the signals into one pass or fail for each task. Examine each failure.
- Examine the results for each risk level.

**Next:** [03 coding agent eval](03%20coding%20agent%20eval.ipynb) (coming soon)